In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import tqdm
import numpy as np

from fugashi import Tagger
from linalgo.annotate import Filter, Sequence2SequenceTransformer, Pipeline
from linalgo.hub import BQClient

from wsd.models import JMDict

In [ ]:
# Fetch documents and annotations from BigQuery
task_id = os.getenv('LINHUB_TASK')
client = BQClient(task_id, project='linalgo-infra')
task = client.get_task()

In [ ]:
# Retrieve the ground truth labels
tagger = Tagger('-Owakati')

def tokenize(text):
    idx = 0
    for token in tagger(text):
        yield idx, token.surface
        idx += len(token.surface)

pipeline = Pipeline([
    Filter(exclude_annotation_fn=lambda a: a.annotator.model == 'MACHINE'),
    Filter(include_document_fn=lambda d: len(d.annotations) > 0),
    Sequence2SequenceTransformer(tokenize_fn=tokenize)
])
input_sequences, output_sequences = pipeline.transform(task)
y_true = np.array([yt for seq in output_sequences for yt in seq])

In [ ]:
jmdict = JMDict()

In [ ]:
# Use the baseline `JMDict` model to disambiguate.
y_pred = []
for doc in tqdm.tqdm(task.documents):
    yp = jmdict.predict(doc.content)
    y_pred.extend(yp)
y_pred = np.array(y_pred)

In [ ]:
# Compute accuracy
acc = np.sum(y_true == y_pred) / len(y_true)
print(f"accuracy = {acc:%}")